In [1]:
import time
import boto3
import pandas as pd

In [11]:
athena = boto3.client(
    "athena",
    endpoint_url="http://localhost:4566",
    region_name="us-east-1",
    aws_access_key_id="test",
    aws_secret_access_key="test",
)

In [15]:
athena.list_query_executions(MaxResults=1)

{'QueryExecutionIds': ['dd285d3f-6801-4825-85dd-2d6e33c08d8a',
  'a9125bdc-1941-45e9-93bc-19625b683b9f',
  '048e3a72-7e44-44b9-b133-5878174e7196',
  '51a80b10-ecc1-43a2-b101-98f6abf22b1c',
  'dc1d7069-b56d-424f-9d82-fd7a1ea6a33f',
  'd407d3ed-b980-4412-8e5c-adde2e353749',
  '0629d43a-d826-47e0-b63a-79e388ccf9c7',
  '5e4e430c-3cf9-4186-8c72-913253d3dbec',
  'aa00ee82-44ca-456e-855e-b1344fb01244',
  '60902c9a-991d-4b16-bd81-0d509d2a88c3',
  '73cd7b4b-3530-476e-a842-0d84c7620e26',
  'ea1bde48-323c-4542-9214-6f4a73dfe3f5',
  '3d588fc8-29df-4300-91ba-1de11b17dfa1'],
 'ResponseMetadata': {'RequestId': '27416cdd-07c1-4917-a182-e2c25d41ff03',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/x-amz-json-1.1;charset=UTF-8',
   'content-length': '530',
   'date': 'Mon, 16 Nov 2026 02:20:43 GMT',
   'x-amz-id-2': '27416cdd-07c1-4917-a182-e2c25d41ff03',
   'x-amz-request-id': '27416cdd-07c1-4917-a182-e2c25d41ff03',
   'x-amzn-requestid': '27416cdd-07c1-4917-a182-e2c25d41ff03'}

In [16]:
def query(sql):
    response = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": "research"},
        ResultConfiguration={
            "OutputLocation": "s3://data-platform-athena-results/queries/"
        },
    )
    query_id = response["QueryExecutionId"]

    while True:
        execution = athena.get_query_execution(QueryExecutionId=query_id)
        status = execution["QueryExecution"]["Status"]
        if status["State"] == "SUCCEEDED":
            break
        if status["State"] in ("FAILED", "CANCELLED"):
            raise RuntimeError(status.get("StateChangeReason") or status["State"])
        time.sleep(0.5)

    pages = athena.get_paginator("get_query_results").paginate(
        QueryExecutionId=query_id
    )
    rows = (
        [cell.get("VarCharValue") for cell in row["Data"]]
        for page in pages
        for row in page["ResultSet"]["Rows"]
    )
    columns = next(rows)  # The first result row contains the column names.
    return pd.DataFrame(rows, columns=columns)

In [18]:
query("SELECT author_type, count(*) as c from prs group by author_type order by author_type")

,author_type,c
0,HUMAN,1
1,Agent,1
2,agent,2
3,human,3
4,unknown,1


In [6]:
# after wer added the second csv file into the s3 bucket
query("SELECT author_type, count(*) as c from prs group by author_type order by author_type")

,author_type,c
0,agent,3
1,human,3


In [9]:
query("select * from prs")

,repo,pr_number,author_type
0,example/demo,7,human
1,example/demo,1,human
2,example/demo,2,agent


In [19]:
query("select * from prs_clean")

,repo,pr_number,author_type
0,example/demo,2,agent
1,example/demo,1,human
